# Stardew Valley Fishing AI — C51 DQN Training on Colab
Uses 4 parallel environments + T4 GPU. Train the model, then download the checkpoint.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# @title 1. Install dependencies & clone repo
import os, sys

BRANCH = "model-upgrade-v2"  # @param {type:"string"}
REPO = "https://github.com/keethesh/StardewValleyFishingAI.git"

if not os.path.exists("StardewValleyFishingAI"):
    !git clone --branch {BRANCH} {REPO}
%cd StardewValleyFishingAI
!git checkout -f {BRANCH}
!git pull origin {BRANCH} 2>/dev/null

# Install deps (no pygame GUI needed)
!pip install -q torch numpy matplotlib

# Tell the script we're on Colab (enables bigger batch size)
os.environ['COLAB_GPU'] = '1'
print(f"Working in: {os.getcwd()}")

In [ ]:
# @title 2. Verify GPU & model
import torch, sys
sys.path.insert(0, '.')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Quick import test
from environment import FishingMinigameEnv
e = FishingMinigameEnv(render_mode=None)
print(f"Env state dim: {e.reset().shape[0]} (expect 24)")

from main import C51DQNAgent, VectorizedEnv
v = VectorizedEnv(num_envs=4, render_mode=None)
a = C51DQNAgent(state_dim=24, action_dim=2)
print(f"Agent params: {sum(p.numel() for p in a.qnetwork_local.parameters()):,}")
print("All imports OK")

In [ ]:
# @title 3. (Optional) Mount Google Drive for persistent model save
# Checkpoints survive Colab disconnects if saved to Drive
from google.colab import drive
import os

MOUNT_DRIVE = False  # @param {type:"boolean"}

if MOUNT_DRIVE:
    drive.mount('/content/drive')
    DRIVE_PATH = '/content/drive/MyDrive/stardew-fishing-models'
    os.makedirs(DRIVE_PATH, exist_ok=True)
    # Symlink models folder to Drive
    if os.path.islink('models'):
        os.unlink('models')
    !rm -rf models
    !ln -sf "{DRIVE_PATH}" models
    print(f"Models will save to: {DRIVE_PATH}")
else:
    !mkdir -p models/checkpoints
    print("Models save locally (lost on disconnect)")

In [ ]:
# @title 4. Start Training!
# Training uses 4 parallel envs + batched GPU forward pass
# Progress prints every 100 episodes, checkpoints every 500

NUM_EPISODES = 12000  # @param {type:"integer"}

os.makedirs("models/checkpoints", exist_ok=True)
os.makedirs("training_logs/graphs", exist_ok=True)

print(f"Starting: {NUM_EPISODES} episodes on 4 parallel envs")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} — batch_size=256, AMP enabled")
print("=" * 60)

!python main.py

In [ ]:
# @title 5. Download trained model + logs
from google.colab import files
import glob, os, zipfile

# Find latest checkpoint
ckpts = sorted(glob.glob("models/checkpoints/*.pth"))
if ckpts:
    latest = ckpts[-1]
    print(f"Latest: {latest}")
    files.download(latest)
else:
    print("No checkpoints yet — training may still be running")

# Also download metrics
for pattern, label in [("training_logs/milestones_*.txt", "Milestones"),
                       ("training_logs/training_metrics_*.csv", "Metrics CSV")]:
    matches = sorted(glob.glob(pattern))
    if matches:
        print(f"{label}: {matches[-1]}")
        files.download(matches[-1])

# Zip all checkpoints for bulk download
zip_path = "all_checkpoints.zip"
with zipfile.ZipFile(zip_path, 'w') as zf:
    for c in ckpts:
        zf.write(c, os.path.basename(c))
print(f"Created {zip_path} ({len(ckpts)} files)")
files.download(zip_path)

In [ ]:
# @title (Optional) Resume from uploaded checkpoint
from google.colab import files

uploaded = files.upload()
for fn in uploaded.keys():
    dest = f"models/checkpoints/{fn}"
    os.makedirs("models/checkpoints", exist_ok=True)
    os.rename(fn, dest)
    print(f"Uploaded -> {dest}")

# Update script for fine-tune mode
with open('main.py', 'r') as f:
    code = f.read()
code = code.replace('train_new_model = True', 'fine_tune_model = True')
code = code.replace("model_path = 'models/YOUR_MODEL_NAME.pth'",
                     f"model_path = '{dest}'")
with open('main.py', 'w') as f:
    f.write(code)
print("Now run cell 4 again to resume training")

In [ ]:
# @title (Optional) Restore from Google Drive after reconnect
from google.colab import drive
import os

DRIVE_PATH = '/content/drive/MyDrive/stardew-fishing-models'

if os.path.isdir('/content/drive/MyDrive'):
    if os.path.exists(DRIVE_PATH):
        !rm -rf models
        !ln -sf "{DRIVE_PATH}" models
        print(f"Linked models -> {DRIVE_PATH}")
        !ls models/checkpoints/
    else:
        print("No Drive backup found — train from scratch")
else:
    # Mount Drive
    drive.mount('/content/drive')
    if os.path.exists(DRIVE_PATH):
        !rm -rf models
        !ln -sf "{DRIVE_PATH}" models
        print(f"Linked models -> {DRIVE_PATH}")
    else:
        print("No backup — create one after training cell 5")